# 01. Collect data from YouTube

**What this notebook does**
1. Reads the list of 9 chosen dramas (`data/dramas_selected.csv`).
2. Finds each drama's full-episode playlist on its official channel and builds a clean episode list.
3. Downloads the newest top-level comments of every episode (up to 500 per episode) with the YouTube Data API.

**Reads:** `data/dramas_selected.csv`
**Writes:** `data/episodes_clean.csv`, `data/raw/comments_<drama>.csv`, `data/youtube_comments.csv`

**Before running:** paste your own YouTube Data API key into the first code cell. Remove it again before sharing this notebook.

Each step skips work that is already saved on disk, so re-running the notebook does not use API quota again.

## 1. Setup

In [ ]:
import os, re, time, glob, warnings
import pandas as pd
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

warnings.filterwarnings("ignore")

API_KEY = "paste your API key here"   # paste your key here, and remove it before submitting
youtube = build("youtube", "v3", developerKey=API_KEY)

os.makedirs("data/raw", exist_ok=True)
CAP = 500                                   # max comments per episode
EPISODES_FILE = "data/episodes_clean.csv"
COMMENTS_FILE = "data/youtube_comments.csv"

## 2. The 9 dramas

Chosen by hand after checking that each one has full episodes on an official channel and that comments are enabled.
Spread: ARY (2 dramas), Hum TV (5), Geo (2). *Kabhi Main Kabhi Tum* was dropped because ARY only uploads clips for it.

In [9]:
dramas = pd.read_csv("data/dramas_selected.csv")[["drama", "video_id"]]
dramas

,drama,video_id
0,Sinf-e-Aahan,RCtj3XNA4FY
1,Jaan Nisar,0TbObX0dyQQ
2,Tere Bin,wykbxHM2Ch4
3,Ishq Murshid,j1j91RAoM8Y
4,Parizaad,vdun73dtG4I
5,Humsafar,iQYGN1_4p08
6,Mere Humsafar,Um-oq3CqceU
7,Ehd-e-Wafa,Gaxsb9Uvwt0
8,Suno Chanda,ODl-DYTyNyM


## 3. Episode list

For each drama we pick its full-episode playlist by title, then:
- take the episode number from the video title (finales called "Last Episode" or "2nd Last Ep" are numbered after the last numbered episode),
- drop non-episodes (promos, OSTs, recaps, behind-the-scenes, full-drama compilations).

Both parts of a split finale keep the same episode number.

In [10]:
# exact playlist title (case and punctuation ignored) for each drama's full episodes
PLAYLIST_TITLES = {
    "Sinf-e-Aahan":  "Sinf e Aahan",
    "Mere Humsafar": "Mere HumSafar",
    "Ishq Murshid":  "Ishq Murshid",
    "Parizaad":      "Parizaad",
    "Humsafar":      "Humsafar - 2022",
    "Ehd-e-Wafa":    "Ehd e Wafa",
    "Suno Chanda":   "Suno Chanda Hum Tv Drama",
    "Jaan Nisar":    "Jaan Nisar",
    "Tere Bin":      "Tere Bin",
}

def norm(s):
    return re.sub(r"[^a-z0-9]+", " ", s.lower()).strip()

def channel_of(video_id):
    r = youtube.videos().list(part="snippet", id=video_id).execute()
    return r["items"][0]["snippet"]["channelId"]

def list_playlists(channel_id, max_pages=40):
    out, token = [], None
    for _ in range(max_pages):
        r = youtube.playlists().list(part="snippet,contentDetails", channelId=channel_id,
                                     maxResults=50, pageToken=token).execute()
        for p in r["items"]:
            out.append({"playlist_id": p["id"], "title": p["snippet"]["title"],
                        "items": p["contentDetails"]["itemCount"]})
        token = r.get("nextPageToken")
        if not token:
            break
    return out

def find_playlist(drama, playlists):
    want = norm(PLAYLIST_TITLES[drama])
    hits = [p for p in playlists if norm(p["title"]) == want]
    if not hits:
        near = [p["title"] for p in playlists if norm(drama).split()[0] in norm(p["title"])]
        raise ValueError(f"No playlist titled '{PLAYLIST_TITLES[drama]}' for {drama}. Similar: {near}")
    return max(hits, key=lambda p: p["items"])

def get_playlist_videos(playlist_id):
    out, token = [], None
    while True:
        r = youtube.playlistItems().list(part="snippet,contentDetails", playlistId=playlist_id,
                                         maxResults=50, pageToken=token).execute()
        for it in r["items"]:
            sn = it["snippet"]
            if sn["title"] in ("Private video", "Deleted video"):
                continue
            out.append({"video_id": sn["resourceId"]["videoId"], "title": sn["title"],
                        "position": sn["position"] + 1,
                        "published": it["contentDetails"].get("videoPublishedAt", "")[:10]})
        token = r.get("nextPageToken")
        if not token:
            break
    return out

def episode_number(title):
    m = re.search(r"\b(?:episode|ep)\b\.?\s*#?\s*(\d+)", title, re.I)
    return int(m.group(1)) if m else None

def clean_episodes(df_eps):
    df = df_eps.copy()
    df["title_l"] = df["title"].str.lower()
    # drop non-episode uploads
    df = df[~df["title_l"].str.contains(r"\b(promo|teaser|ost)\b", regex=True)]
    junk_unnumbered = r"\b(recap|weekly|roundup|behind the scene|spotlight|full episode)\b"
    df = df[~(df["episode"].isna() & df["title_l"].str.contains(junk_unnumbered, regex=True))]
    # number the "2nd last" and "last" episodes
    parts = []
    for drama, g in df.groupby("drama", sort=False):
        g = g.copy()
        n = g["episode"].max()
        second_last = g["episode"].isna() & g["title_l"].str.contains("2nd last")
        last = g["episode"].isna() & g["title_l"].str.contains("last") & ~second_last
        if second_last.any():
            n += 1
            g.loc[second_last, "episode"] = n
        if last.any():
            n += 1
            g.loc[last, "episode"] = n
        parts.append(g)
    out = pd.concat(parts).drop(columns="title_l")
    out["episode"] = out["episode"].astype(int)
    return out.sort_values(["drama", "episode", "position"]).reset_index(drop=True)

In [11]:
if os.path.exists(EPISODES_FILE):
    df_clean = pd.read_csv(EPISODES_FILE)
    print("Loaded existing", EPISODES_FILE)
else:
    channel_playlists = {}
    rows = []
    for _, d in dramas.iterrows():
        cid = channel_of(d["video_id"])
        if cid not in channel_playlists:
            channel_playlists[cid] = list_playlists(cid)
        pl = find_playlist(d["drama"], channel_playlists[cid])
        print(f"{d['drama']:15s} -> '{pl['title']}' ({pl['items']} videos)")
        for v in get_playlist_videos(pl["playlist_id"]):
            rows.append({"drama": d["drama"], **v})
    df_eps = pd.DataFrame(rows)
    df_eps["episode"] = df_eps["title"].apply(episode_number)
    df_clean = clean_episodes(df_eps)
    df_clean.to_csv(EPISODES_FILE, index=False)
    print("Saved", EPISODES_FILE)

Loaded existing data/episodes_clean.csv


**Check:** `missing_eps` should be 0 for every drama. Two videos can share a number when a finale is split in two parts.

In [12]:
summary = df_clean.groupby("drama").agg(videos=("video_id", "count"),
                                        unique_eps=("episode", "nunique"),
                                        last_ep=("episode", "max"))
summary["missing_eps"] = summary["last_ep"] - summary["unique_eps"]
summary

,videos,unique_eps,last_ep,missing_eps
drama,,,,
Ehd-e-Wafa,25,25,25,0
Humsafar,23,23,23,0
Ishq Murshid,32,31,31,0
Jaan Nisar,65,65,65,0
Mere Humsafar,40,40,40,0
Parizaad,30,29,29,0
Sinf-e-Aahan,23,23,23,0
Suno Chanda,30,30,30,0
Tere Bin,58,58,58,0


## 4. Download comments

- Newest comments first (`order="time"`). The default "relevance" order shows the most-liked comments, which would lean positive.
- Top-level comments only, up to 500 per episode.
- One CSV per drama in `data/raw/`. If the notebook stops (for example the daily quota ends), run it again and finished dramas are skipped.
- The author name is not saved.

In [13]:
def fetch_comments(video_id, cap=CAP):
    out, token = [], None
    while len(out) < cap:
        try:
            r = youtube.commentThreads().list(part="snippet", videoId=video_id, maxResults=100,
                                              order="time", textFormat="plainText",
                                              pageToken=token).execute()
        except HttpError as e:
            if "commentsDisabled" in str(e):
                return out          # comments are off on this video, fine
            raise                   # anything else (bad key, quota) must stop the run
        for it in r["items"]:
            c = it["snippet"]["topLevelComment"]["snippet"]
            out.append({"comment_id": it["id"], "text": c["textDisplay"],
                        "likes": c["likeCount"], "published": c["publishedAt"]})
        token = r.get("nextPageToken")
        if not token:
            break
    return out[:cap]

for drama, grp in df_clean.groupby("drama", sort=False):
    fname = f"data/raw/comments_{drama.replace(' ', '_')}.csv"
    if os.path.exists(fname) and os.path.getsize(fname) > 100:
        print(f"skip {drama} (already saved)")
        continue
    rows = []
    try:
        for _, ep in grp.sort_values(["episode", "position"]).iterrows():
            for c in fetch_comments(ep["video_id"]):
                rows.append({"drama": drama, "episode": ep["episode"], "video_id": ep["video_id"], **c})
            print(f"{drama} ep {ep['episode']}: total {len(rows)}")
            time.sleep(0.2)
    except HttpError as e:
        print("Stopped:", str(e)[:200])
        break
    if rows:
        pd.DataFrame(rows).to_csv(fname, index=False)
        print(f"saved {fname} ({len(rows)} comments)")

skip Ehd-e-Wafa (already saved)
skip Humsafar (already saved)
skip Ishq Murshid (already saved)
skip Jaan Nisar (already saved)
skip Mere Humsafar (already saved)
skip Parizaad (already saved)
skip Sinf-e-Aahan (already saved)
skip Suno Chanda (already saved)
Tere Bin ep 1: total 500
Tere Bin ep 2: total 1000
Tere Bin ep 3: total 1500
Tere Bin ep 4: total 2000
Tere Bin ep 5: total 2500
Tere Bin ep 6: total 3000
Tere Bin ep 7: total 3500
Tere Bin ep 8: total 4000
Tere Bin ep 9: total 4500
Tere Bin ep 10: total 5000
Tere Bin ep 11: total 5500
Tere Bin ep 12: total 6000
Tere Bin ep 13: total 6500
Tere Bin ep 14: total 7000
Tere Bin ep 15: total 7500
Tere Bin ep 16: total 8000
Tere Bin ep 17: total 8500
Tere Bin ep 18: total 9000
Tere Bin ep 19: total 9500
Tere Bin ep 20: total 10000
Tere Bin ep 21: total 10500
Tere Bin ep 22: total 11000
Tere Bin ep 23: total 11500
Tere Bin ep 24: total 12000
Tere Bin ep 25: total 12500
Tere Bin ep 26: total 13000
Tere Bin ep 27: total 13500
Tere Bin ep 2

## 5. Combine all dramas into one file

In [14]:
files = [f for f in glob.glob("data/raw/comments_*.csv") if os.path.getsize(f) > 100]
print(len(files), "usable files (expected 9)")

df_comments = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
df_comments = df_comments.drop(columns="author", errors="ignore").drop_duplicates("comment_id")
df_comments.to_csv(COMMENTS_FILE, index=False)
print(len(df_comments), "comments from", df_comments["drama"].nunique(), "dramas")

df_comments.groupby("drama").agg(comments=("comment_id", "count"), episodes=("episode", "nunique"))

9 usable files (expected 9)
159233 comments from 9 dramas


,comments,episodes
drama,,
Ehd-e-Wafa,12500,25
Humsafar,7733,23
Ishq Murshid,16000,31
Jaan Nisar,32500,65
Mere Humsafar,20000,40
Parizaad,15000,29
Sinf-e-Aahan,11500,23
Suno Chanda,15000,30
Tere Bin,29000,58
